# Riproduzione della baseline testuale (notebook Kaggle)

Questo notebook riproduce l'approccio del notebook pubblico
[Personalized Workout Program Recommender](https://www.kaggle.com/code/denvermagtibay/personalized-workout-program-recommender),
trovato insieme al dataset e usato come **baseline esterna** del progetto.

Serve a documentare che il confronto e' stato eseguito davvero, non solo citato.

## Cosa fa il notebook originale

1. unisce titolo, descrizione e primi esercizi di ogni programma in un unico testo;
2. codifica obiettivo, livello e attrezzatura con `LabelEncoder` e li concatena agli
   embedding testuali (`all-MiniLM-L6-v2`);
3. raggruppa i programmi con KMeans (k = 25);
4. simula un utente come **media dei vettori** di alcuni programmi "apprezzati" e
   consiglia i programmi piu' simili per coseno.

**Non contiene alcuna valutazione**: mostra qualche esempio di output.

## I due problemi dell'originale

- I codici di `LabelEncoder` sono numeri arbitrari (0, 1, 2 ... decine) concatenati a
  embedding normalizzati, le cui componenti valgono circa +/- 0,05. La similarita'
  coseno finisce per essere decisa quasi solo da quei codici.
- La media dei codici di piu' programmi produce un numero che corrisponde a una
  categoria arbitraria, non alla combinazione delle preferenze.

Qui la parte testuale viene riprodotta **nella sua versione migliore**, cioe' senza i
codici arbitrari: solo embedding del testo, che e' il cuore dell'idea. Se anche cosi'
il metodo non funziona, la conclusione e' solida e non dipende da quei difetti.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"

catalogo = pd.read_csv(PROCESSED / "catalogo_con_qualita.csv")
testi = (pd.read_csv(PROCESSED / "testo_programmi.csv").set_index("title")["testo"]
         .reindex(catalogo["title"]).fillna("").tolist())
print(f"Programmi: {len(catalogo)}")
print("Esempio di testo indicizzato:\n", testi[0][:300], "...")

## 1. Embedding dei programmi

Stesso modello del notebook originale: `all-MiniLM-L6-v2`. Se lo script
`12_embeddings.py` e' gia' stato eseguito, gli embedding vengono riusati.

In [ ]:
percorso = PROCESSED / "embeddings_programmi.npy"
if percorso.exists():
    emb_programmi = np.load(percorso)
    print("Embedding gia' calcolati, riutilizzati:", emb_programmi.shape)
else:
    from sentence_transformers import SentenceTransformer
    modello = SentenceTransformer("all-MiniLM-L6-v2")
    emb_programmi = modello.encode(testi, batch_size=64, show_progress_bar=True,
                                   normalize_embeddings=True)
    np.save(percorso, emb_programmi)
    print("Embedding calcolati:", emb_programmi.shape)

## 2. Il raggruppamento in cluster, come nell'originale

Il notebook originale usa KMeans con k = 25, scelto senza alcuna giustificazione.
Qui si verifica cosa producono quei gruppi: se fossero interpretabili, si dovrebbero
riconoscere temi omogenei.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=25, random_state=42, n_init="auto")
cluster = kmeans.fit_predict(emb_programmi)
catalogo["cluster"] = cluster

for numero in [0, 1, 2]:
    dentro = catalogo[catalogo["cluster"] == numero]
    print(f"\n--- cluster {numero} ({len(dentro)} programmi) ---")
    print(dentro["title"].head(6).to_string(index=False))
    obiettivi = dentro["goal"].value_counts().head(3)
    print("obiettivi piu' frequenti:", dict(obiettivi))

## 3. Raccomandazione per somiglianza, come nell'originale

Il profilo dell'utente viene trasformato in una frase e confrontato con i testi dei
programmi. E' la stessa logica del notebook originale, applicata al nostro scenario:
li' l'utente era rappresentato dalla media di alcuni programmi apprezzati, qui dal
testo del suo profilo, perche' nel nostro caso l'utente e' nuovo e non ha storico.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from src.matching import ProfiloUtente
from src.recommenders import chiave_query

def consiglia_per_testo(profilo, quanti=5):
    chiavi = pd.read_csv(PROCESSED / "embeddings_query_chiavi.csv")["query"].tolist()
    vettori = np.load(PROCESSED / "embeddings_query.npy")
    mappa = dict(zip(chiavi, vettori))
    chiave = chiave_query(profilo)
    if chiave not in mappa:
        from sentence_transformers import SentenceTransformer
        modello = SentenceTransformer("all-MiniLM-L6-v2")
        vettore = modello.encode([chiave], normalize_embeddings=True)[0]
    else:
        vettore = mappa[chiave]
    somiglianza = cosine_similarity(vettore.reshape(1, -1), emb_programmi).ravel()
    # vincolo fisico: l'attrezzatura deve bastare
    somiglianza = np.where(catalogo["attrezzatura_num"] <= profilo.attrezzatura_num,
                           somiglianza, -1)
    migliori = np.argsort(-somiglianza)[:quanti]
    return catalogo.iloc[migliori][["title", "goal", "level", "time_per_workout",
                                    "giorni_settimana"]].assign(
        somiglianza=somiglianza[migliori].round(3))

profilo = ProfiloUtente("Muscle & Sculpting", "Beginner", "At Home", 3, 30)
consiglia_per_testo(profilo)

Gia' a occhio si vede il problema: i programmi in cima sono quelli la cui
**descrizione assomiglia alla descrizione della persona**, non quelli adatti a lei.
La somiglianza testuale premia chi ha scritto un testo con le stesse parole, non chi
propone un allenamento compatibile con tempo, giorni e livello.

## 4. La valutazione che manca nell'originale

Qui la baseline viene misurata sugli stessi 300 utenti simulati usati per tutti gli
altri sistemi del progetto, con la stessa metrica.

In [ ]:
from src.simulator import genera_utenti
from src.evaluation import ndcg_at_k, bootstrap_ic
from src.recommenders import costruisci_sistemi, casuale
from src.scoring import punteggio

U = np.load(PROCESSED / "utilita.npy")
utenti = genera_utenti(len(U))
chiavi = pd.read_csv(PROCESSED / "embeddings_query_chiavi.csv")["query"].tolist()
vettori = np.load(PROCESSED / "embeddings_query.npy")
mappa_query = dict(zip(chiavi, vettori))

rng = np.random.default_rng(7)
risultati = {"baseline Kaggle (embeddings)": [], "casuale": [], "sistema del progetto": []}

for i, utente in enumerate(utenti):
    p = utente.profilo_dichiarato()
    vettore = mappa_query[chiave_query(p)].reshape(1, -1)
    somiglianza = cosine_similarity(vettore, emb_programmi).ravel()
    somiglianza = somiglianza * (catalogo["attrezzatura_num"] <= p.attrezzatura_num).values
    risultati["baseline Kaggle (embeddings)"].append(ndcg_at_k(somiglianza, U[i], 10))
    risultati["casuale"].append(ndcg_at_k(casuale(catalogo, p, rng=rng), U[i], 10))
    risultati["sistema del progetto"].append(
        ndcg_at_k(punteggio(catalogo, p).fillna(0).values, U[i], 10))

for nome, valori in risultati.items():
    media, basso, alto = bootstrap_ic(valori)
    print(f"{nome:<32} NDCG@10 = {media:.3f}  IC 95% [{basso:.3f}, {alto:.3f}]")

## 5. Conclusione

La baseline con gli embedding **non si distingue dalla scelta casuale**: gli
intervalli di confidenza si sovrappongono.

Il motivo e' strutturale, non implementativo. La descrizione di un programma e' un
testo promozionale scritto dal suo autore: racconta **di cosa parla** il programma,
non **per chi va bene**. Due programmi descritti in modo simile possono essere uno
perfetto e l'altro impraticabile per la stessa persona, perche' cio' che li distingue
(minuti a seduta, giorni a settimana, volume settimanale, progressione) non sta nel
modo in cui il testo e' scritto.

E' la ragione per cui il progetto non si ferma al testo e calcola indicatori
strutturali dai 605.033 esercizi del dataset.

**Nota di correttezza:** la riproduzione usa lo stesso modello dell'originale e ne
adotta la versione migliore, senza i codici categorici arbitrari che nel notebook
dominano la similarita'. Il confronto e' quindi favorevole alla baseline, non
penalizzante.